Ik wil een aantal dingen onderzoeken:
- Temperatuur van landen op een wereldkaart (kleur geeft temperatuur)
- Onzekerheid van landen op een wereldkaart (kleur geeft onzekerheid)


In [1]:
import matplotlib.pyplot as plt
import plotly.express as px
import pandas as pd
from sqlalchemy import create_engine

In [2]:
engine = create_engine(
    "mysql+pymysql://root@localhost:3306/climate_watch"
)

df_country = pd.read_sql("SELECT * FROM GlobalLandTemperaturesByCountry", con=engine, parse_dates=["dt"])
df_city = pd.read_sql("SELECT * FROM GlobalLandTemperaturesByCity", con=engine, parse_dates=["dt"])

## Scatter Geo

In [ ]:
def convert_coordinate_to_numeric(coord: str) -> float:
    value = float(coord[:-1])
    direction = coord[-1]

    if direction in ["S", "W"]:
        value = -value

    return value


date = "1950-01-01"
df_filter_date = df_city[df_city["dt"] == date].dropna()
df_filter_date["Latitude_num"] = df_filter_date["Latitude"].apply(convert_coordinate_to_numeric)
df_filter_date["Longitude_num"] = df_filter_date["Longitude"].apply(convert_coordinate_to_numeric)
fig = px.scatter_geo(
    data_frame=df_filter_date, 
    lat="Latitude_num", 
    lon="Longitude_num",
    color="AverageTemperature",
    hover_name="City",
    hover_data="Country",
    color_continuous_scale="RdBu_r",
    range_color=(-40,40),
    title=f"Average Temperature (°C) per City on {date}",
    )
fig.show()

In [ ]:
date = "1950-01-01"
df_filter_date = df_country[df_country["dt"] == date].dropna()

fig = px.scatter_geo(
    data_frame=df_filter_date, 
    color="AverageTemperature",
    hover_name="Country",
    color_continuous_scale="RdBu_r",
    range_color=(-40,40),
    title=f"Average Temperature (°C) per City on {date}",
    locations="Country",
    locationmode="country names"
    )
fig.show()

## Choropleth map

In [78]:
date = "2000-01-01"
df_filter_date = df_country[df_country["dt"] == date].dropna()

fig = px.choropleth(
    data_frame=df_filter_date,
    locations="Country",
    locationmode="country names",
    color="AverageTemperature",
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0,
    range_color=(-40,40),
    title= f"Average Temperature (°C) per Country on {date}"
)

fig.update_geos(
    showland=True,
    landcolor="#5a5b5c",
    showocean=True,
    oceancolor="#c3daf1",
    showlakes=False
    )

fig.show()

In [98]:
date = "2000-01-01"
df_filter_date = df_country[df_country["dt"] == date].dropna()

color_scale = [
    (0, "lightgray"),
    (1, "red")
]

fig = px.choropleth(
    data_frame=df_filter_date,
    locations="Country",
    locationmode="country names",
    color="AverageTemperatureUncertainty",
    color_continuous_scale=color_scale,
    # range_color=(0,4),
    title= f"Average Temperature Uncertainty (°C) per Country on {date}"
)

fig.update_geos(
    showland=True,
    landcolor="#5a5b5c",
    showocean=True,
    oceancolor="#c3daf1",
    showlakes=False,
)

fig.show()